In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import pymn
import anndata as ad
import time
import os
from pyprojroot import here
import resource
import scipy.sparse as sp


/home/leon/miniconda3/envs/mouse_brain_cells/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/leon/miniconda3/envs/mouse_brain_cells/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
base_data_folder = "/vault/lfrench/temp_transfer/retina_cellxgene/"

In [3]:
#adata_retina = sc.read_h5ad(base_data_folder + "b4b58102-d03b-47b9-93b1-ed419b042de7.h5ad")

In [4]:
adata_retina = sc.read_h5ad(base_data_folder + "6eb388aa-9aa4-41a6-8d87-55385a4faea2.h5ad")

In [5]:
adata_retina

AnnData object with n_obs × n_vars = 3177310 × 35475
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_enriched_cell_types', 'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'reported_diseases', 'sex_ontology_term_id', 'majorclass', 'AC_subclass', 'AC_cluster', 'AC_celltype_number', 'BC_subclass',

In [6]:
adata_retina.obs['cell.type'] = adata_retina.obs['author_cell_type'].astype(str)
adata_retina.obs['study_id'] = "LiRetina"
adata_retina.obs.index = adata_retina.obs.index.to_numpy(dtype="str")

In [7]:
print("Number of adata_retina cell types: " + str(len(set(adata_retina.obs["cell.type"]))))


Number of adata_retina cell types: 123


In [14]:
set(adata_retina.obs["cell.type"])

{'AII',
 'AII*',
 'Astrocyte',
 'BB',
 'CAI',
 'CAII',
 'DB1',
 'DB2',
 'DB3a',
 'DB3b',
 'DB4a',
 'DB4b',
 'DB5',
 'DB6',
 'FMB',
 'GB',
 'H1',
 'H2',
 'HAC11',
 'HAC12',
 'HAC13',
 'HAC14',
 'HAC15',
 'HAC17',
 'HAC18',
 'HAC19',
 'HAC2',
 'HAC20',
 'HAC21',
 'HAC23',
 'HAC24',
 'HAC26',
 'HAC27',
 'HAC28',
 'HAC29',
 'HAC3',
 'HAC30',
 'HAC32',
 'HAC33',
 'HAC34',
 'HAC35',
 'HAC37',
 'HAC38',
 'HAC39',
 'HAC4',
 'HAC40',
 'HAC41',
 'HAC42',
 'HAC44',
 'HAC45',
 'HAC46',
 'HAC48',
 'HAC49',
 'HAC5',
 'HAC50',
 'HAC51',
 'HAC52',
 'HAC53',
 'HAC54',
 'HAC55',
 'HAC56',
 'HAC57',
 'HAC59',
 'HAC6',
 'HAC60',
 'HAC62',
 'HAC63',
 'HAC64',
 'HAC65',
 'HAC66',
 'HAC68',
 'HAC71',
 'HAC72',
 'HAC74',
 'HAC75',
 'HAC77',
 'HAC78',
 'HAC79',
 'HAC8',
 'HAC80',
 'HAC81',
 'HAC82',
 'HAC84',
 'HAC86',
 'HAC87',
 'HAC88',
 'HAC89',
 'HAC9',
 'HAC90',
 'HAC91',
 'HAC92',
 'HAC93',
 'HAC95',
 'HRGC10',
 'HRGC11',
 'HRGC3',
 'HRGC4',
 'HRGC5',
 'HRGC9',
 'IMB',
 'MG',
 'MG_OFF',
 'MG_ON',
 'ML_Co

In [8]:
adata_retina

AnnData object with n_obs × n_vars = 3177310 × 35475
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_enriched_cell_types', 'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'reported_diseases', 'sex_ontology_term_id', 'majorclass', 'AC_subclass', 'AC_cluster', 'AC_celltype_number', 'BC_subclass',

In [9]:

# Basic inspection for whether adata_retina looks like raw counts or normalized/transformed data

def sample_values(x, n=100000, random_state=0):
    rng = np.random.default_rng(random_state)
    if sp.issparse(x):
        data = x.data
    else:
        data = np.asarray(x).ravel()
    if data.size == 0:
        return data
    if data.size > n:
        idx = rng.choice(data.size, size=n, replace=False)
        data = data[idx]
    return data

def check_matrix(name, x):
    vals = sample_values(x)

    if vals.size == 0:
        print(f"\n{name}: empty")
        return

    finite = vals[np.isfinite(vals)]
    frac_integer = np.mean(np.isclose(finite, np.round(finite)))
    has_negative = np.any(finite < 0)
    has_non_integer = np.any(~np.isclose(finite, np.round(finite)))
    vmax = np.max(finite)
    vmin = np.min(finite)

    # sample row sums
    n_rows = x.shape[0]
    rows_to_check = min(1000, n_rows)
    row_idx = np.random.choice(n_rows, size=rows_to_check, replace=False)

    if sp.issparse(x):
        row_sums = np.asarray(x[row_idx].sum(axis=1)).ravel()
    else:
        row_sums = np.asarray(x[row_idx].sum(axis=1)).ravel()

    print(f"\n{name}")
    print("-" * len(name))
    print("shape:", x.shape)
    print("min/max sampled values:", vmin, vmax)
    print("fraction integer-like (sampled):", round(frac_integer, 4))
    print("any negative values:", has_negative)
    print("any non-integer values:", has_non_integer)
    print("sample row sums: min/median/max =",
          round(float(np.min(row_sums)), 3),
          round(float(np.median(row_sums)), 3),
          round(float(np.max(row_sums)), 3))

    # crude interpretation
    if has_negative:
        print("Interpretation: not raw counts; likely scaled/log-transformed data.")
    elif frac_integer > 0.99:
        print("Interpretation: this strongly looks like raw count data.")
    else:
        print("Interpretation: this looks more like normalized and/or log-transformed data.")

# Check main matrix
check_matrix("adata_retina.X", adata_retina.X)

# Check raw, if present
if adata_retina.raw is not None:
    check_matrix("adata_retina.raw.X", adata_retina.raw.X)

# Check layers, if present
if len(adata_retina.layers) > 0:
    print("\nLayers found:", list(adata_retina.layers.keys()))
    for layer_name in adata_retina.layers.keys():
        check_matrix(f"adata_retina.layers['{layer_name}']", adata_retina.layers[layer_name])

# Also inspect common metadata clues
print("\nobs columns:", list(adata_retina.obs.columns[:20]))
print("var columns:", list(adata_retina.var.columns[:20]))
print("uns keys:", list(adata_retina.uns.keys())[:20])

# Common Scanpy clues
for key in ["log1p", "normalize_total"]:
    if key in adata_retina.uns:
        print(f"Found adata_retina.uns['{key}'] -> suggests normalization/log transform was applied")


adata_retina.X
--------------
shape: (3177310, 35475)
min/max sampled values: 0.0 6.967968
fraction integer-like (sampled): 0.0445
any negative values: False
any non-integer values: True
sample row sums: min/median/max = 1043.976 2052.066 2734.209
Interpretation: this looks more like normalized and/or log-transformed data.

adata_retina.raw.X
------------------
shape: (3177310, 35475)
min/max sampled values: 1.0 1047.0
fraction integer-like (sampled): 1.0
any negative values: False
any non-integer values: False
sample row sums: min/median/max = 544.0 4635.0 31667.0
Interpretation: this strongly looks like raw count data.

obs columns: ['reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_met

In [10]:
adata_retina = adata_retina.raw.to_adata()

In [11]:
adata_retina

AnnData object with n_obs × n_vars = 3177310 × 35475
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_enriched_cell_types', 'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'reported_diseases', 'sex_ontology_term_id', 'majorclass', 'AC_subclass', 'AC_cluster', 'AC_celltype_number', 'BC_subclass',

In [12]:
rng = np.random.default_rng(42)

n_cells = adata_retina.n_obs
perm = rng.permutation(n_cells)
split_point = n_cells // 2

idx1 = perm[:split_point]
idx2 = perm[split_point:]

# Sort indices just to keep cell order tidy within each subset
idx1 = np.sort(idx1)
idx2 = np.sort(idx2)

# Subset
adata_retina_1 = adata_retina[idx1].copy()
adata_retina_1.obs['study_id'] = "LiRetina_A"

adata_retina_2 = adata_retina[idx2].copy()
adata_retina_2.obs['study_id'] = "LiRetina_B"


print(adata_retina_1)
print(adata_retina_2)



AnnData object with n_obs × n_vars = 1588655 × 35475
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_enriched_cell_types', 'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'reported_diseases', 'sex_ontology_term_id', 'majorclass', 'AC_subclass', 'AC_cluster', 'AC_celltype_number', 'BC_subclass',

In [13]:
adata_retina_1.write_h5ad(base_data_folder + "LiRetina_A.h5ad")
adata_retina_2.write_h5ad(base_data_folder + "LiRetina_B.h5ad")